In [1]:
from typing import Dict, Any
import torch
from monai.networks.nets import unetr
from pytorch3dunet.datasets.hdf5 import StandardHDF5Dataset

from model_ranking.models import UnetrWrapper

In [14]:
dataset_config: Dict[str, Any] = {
    #"file_path": "/scratch/talks/data/EPFL/train.h5",
    "file_path": "/scratch/talks/data/EPFL/val.h5",
    #"file_path": "/scratch/talks/data/VNC/resized_pixels/source_mitoEM_true.h5",
    "roi" : None,
    #"roi": [[0,50], [1024, 2048], [1024, 2048]],
    #"roi": [[0,300], [0,1024], [0,1024]],
    "phase": "train",
    "slice_builder_config": {
        "name": "SliceBuilder",
        "patch_shape": [1, 256, 256],
        "stride_shape": [4, 256, 256],
        "halo_shape": [0, 32, 32],
    },
    "transformer_config": {
        "raw" : [{"name": "Normalize"}, {"name": "ToTensor", "expand_dims": True}],
        "label": [{"name": "ToTensor", "expand_dims": True}],
    },
    "raw_internal_path": "raw",
    "label_internal_path": "labels",
    "global_normalization": False,
}

In [15]:
dataset = StandardHDF5Dataset(**dataset_config)

2025-07-23 17:03:45,216 [MainThread] INFO Dataset - Slice builder config: {'name': 'SliceBuilder', 'patch_shape': [1, 256, 256], 'stride_shape': [4, 256, 256], 'halo_shape': [0, 32, 32]}
2025-07-23 17:03:45,220 [MainThread] INFO HDF5Dataset - Number of patches: 66


In [23]:
5775/66 *60

5250.0

In [22]:
88/4

22.0

In [20]:
5775/66

87.5

In [4]:
model = unetr.UNETR(in_channels=1, out_channels=1, img_size=256, feature_size=32, norm_name='batch', spatial_dims=2)
model.eval()

UNETR(
  (vit): ViT(
    (patch_embedding): PatchEmbeddingBlock(
      (patch_embeddings): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (mlp): MLPBlock(
          (linear1): Linear(in_features=768, out_features=3072, bias=True)
          (linear2): Linear(in_features=3072, out_features=768, bias=True)
          (fn): GELU(approximate='none')
          (drop1): Dropout(p=0.0, inplace=False)
          (drop2): Dropout(p=0.0, inplace=False)
        )
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SABlock(
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
          (qkv): Linear(in_features=768, out_features=2304, bias=False)
          (to_q): Identity()
          (to_k): Identity()
          (to_v): Identity()
          (input_rearrange): Rearrange('b h (qkv l d) -> qkv b l h d', qkv=3, l=12)
 

In [5]:
model_unetr = UnetrWrapper(
    in_channels=1, 
    out_channels=1, 
    img_size=256, 
    feature_size=32, 
    norm_name='batch',
    spatial_dims=2,
)
model_unetr.eval()

UnetrWrapper(
  (vit): ViT(
    (patch_embedding): PatchEmbeddingBlock(
      (patch_embeddings): Conv2d(1, 768, kernel_size=(16, 16), stride=(16, 16))
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (blocks): ModuleList(
      (0-11): 12 x TransformerBlock(
        (mlp): MLPBlock(
          (linear1): Linear(in_features=768, out_features=3072, bias=True)
          (linear2): Linear(in_features=3072, out_features=768, bias=True)
          (fn): GELU(approximate='none')
          (drop1): Dropout(p=0.0, inplace=False)
          (drop2): Dropout(p=0.0, inplace=False)
        )
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): SABlock(
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
          (qkv): Linear(in_features=768, out_features=2304, bias=False)
          (to_q): Identity()
          (to_k): Identity()
          (to_v): Identity()
          (input_rearrange): Rearrange('b h (qkv l d) -> qkv b l h d', qkv=3, 

In [6]:
with torch.no_grad():
    for i in range(10):
        raw, label = dataset[i]
        pred = model_unetr(raw.unsqueeze(0))
        print(pred.shape)
        print(pred.min(), pred.max())

torch.Size([1, 1, 1, 256, 256])
tensor(0.4602) tensor(0.5142)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4589) tensor(0.5143)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4603) tensor(0.5143)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4564) tensor(0.5143)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4587) tensor(0.5144)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4588) tensor(0.5140)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4642) tensor(0.5146)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4557) tensor(0.5143)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4620) tensor(0.5139)
torch.Size([1, 1, 1, 256, 256])
tensor(0.4564) tensor(0.5143)


In [2]:
model_config: Dict[str, Any] = {
    "name": "UnetrWrapper",
    "in_channels": 1,
    "out_channels": 1,
    "img_size": 256,  # Place holder size will be overwritten on creation of UnetrModelConfig
    "feature_size": 16,
    "hidden_size": 768,
    "mlp_dim": 3072,
    "num_heads": 12,
    "proj_type": "conv",
    "norm_name": "batch",
    "conv_block": True,
    "res_block": True,
    "dropout_rate": 0.0,
    "spatial_dims": 2,
    "qkv_bias": False,
    "save_attn": False,
    "is_segmentation": True,
    "final_sigmoid": True,
}

In [3]:
from pytorch3dunet.unet3d.model import get_model

model_unetr = get_model(
    model_config
)